# 決算反応モデル — 予測 & 答え合わせ

### 使い方

**Step 1: 次のセルで日付を設定**

**Step 2: 予測（ザラバ + 引け後）**
- `PREDICT_DATE` を当日に設定 → `Ctrl+F9`（全セル実行）
- ザラバ銘柄（`is_intraday=True`）と引け後銘柄を一括スコアリング

**Step 3: 答え合わせ**
- ザラバ銘柄: 当日終値 vs 前日終値（当日中に実行可能）
- 引け後銘柄: `ACTUAL_DATE` を翌営業日に設定 → Step 3 セルのみ実行


In [ ]:
# ╔══════════════════════════════════════════════╗
# ║  ★ ここだけ変更する ★                        ║
# ╚══════════════════════════════════════════════╝

# 予測対象日（決算発表日）
PREDICT_DATE = '20260406'  # ← 当日の日付に変更

# 答え合わせ日（翌営業日）— Step 3 の引け後銘柄で使用
ACTUAL_DATE = '20260407'   # ← 翌営業日の日付に変更

# ── 以下は変更不要 ──
PREDICT_DATE_HYPHEN = f'{PREDICT_DATE[:4]}-{PREDICT_DATE[4:6]}-{PREDICT_DATE[6:8]}'
ACTUAL_DATE_HYPHEN = f'{ACTUAL_DATE[:4]}-{ACTUAL_DATE[4:6]}-{ACTUAL_DATE[6:8]}'
print(f'予測対象日: {PREDICT_DATE_HYPHEN}  答え合わせ日: {ACTUAL_DATE_HYPHEN}')


In [ ]:
# ── Colab 依存パッケージ ──
%pip install -q jquants-api-client


In [ ]:
%matplotlib inline
import os, sys, json, time
from pathlib import Path
from datetime import datetime, timedelta, date
from zoneinfo import ZoneInfo
JST = ZoneInfo("Asia/Tokyo")
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
try:
    import japanize_matplotlib
except ImportError:
    import subprocess
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'japanize-matplotlib'], capture_output=True)
    import japanize_matplotlib
matplotlib.rcParams['axes.unicode_minus'] = False
import requests
import warnings
warnings.filterwarnings('ignore')

# ── 実行環境判定 ──
try:
    from google.colab import auth, userdata
    RUNTIME = 'colab'
except ImportError:
    RUNTIME = 'local'

if RUNTIME == 'colab':
    auth.authenticate_user()
    from google.cloud import bigquery, storage
    bq = bigquery.Client(project='gmailpj-357912')
    gcs = storage.Client(project='gmailpj-357912')
    try:
        JQUANTS_API_KEY = userdata.get('JQUANTS_API_KEY')
    except Exception:
        JQUANTS_API_KEY = os.environ.get('JQUANTS_API_KEY', '')
        if not JQUANTS_API_KEY:
            print('WARN: JQUANTS_API_KEY not found in Colab Secrets. Set it in left sidebar > Secrets.')
else:
    import urllib3, requests as _req
    from requests.adapters import HTTPAdapter as _HA
    urllib3.disable_warnings()
    class _NoVerify(_HA):
        def send(self, req, **kw): kw['verify'] = False; return super().send(req, **kw)
    _orig = _req.Session.__init__
    def _p(self, *a, **kw): _orig(self, *a, **kw); self.mount('https://', _NoVerify()); self.verify = False
    _req.Session.__init__ = _p

    PROJECT_ROOT = Path(r'C:\gdrive\claude\investment-agent')
    sys.path.insert(0, str(PROJECT_ROOT / 'scripts'))
    from dotenv import load_dotenv
    load_dotenv(PROJECT_ROOT / '.env')

    from google.cloud import bigquery, storage
    from google.oauth2 import service_account
    KEY_FILE = str(PROJECT_ROOT / 'keys' / 'gcp-service-account.json')
    creds = service_account.Credentials.from_service_account_file(KEY_FILE)
    bq = bigquery.Client(credentials=creds, project='gmailpj-357912')
    gcs = storage.Client(credentials=creds, project='gmailpj-357912')
    JQUANTS_API_KEY = os.environ['JQUANTS_API_KEY']


# GCS パス
GCS_BUCKET = 'gs://stock_data_1930932'
GCS_BUCKET_NAME = 'stock_data_1930932'
GCS_PREDICTIONS = f'{GCS_BUCKET}/earnings_model/predictions'
GCS_ACTUALS = f'{GCS_BUCKET}/earnings_model/actuals'
GCS_ACCURACY = f'{GCS_BUCKET}/earnings_model/accuracy'



def gcs_save_json(data: dict | list, gcs_path: str) -> None:
    """Save JSON to GCS.

    Args:
        data: Data to serialize as JSON.
        gcs_path: Full GCS path (gs://bucket/path/file.json).
    """
    path = gcs_path.replace(f'gs://{GCS_BUCKET_NAME}/', '')
    bucket = gcs.bucket(GCS_BUCKET_NAME)
    blob = bucket.blob(path)
    blob.upload_from_string(
        json.dumps(data, ensure_ascii=False, indent=2, default=str),
        content_type='application/json'
    )
    print(f'Saved to {gcs_path}')


def gcs_load_json(gcs_path: str) -> dict | list | None:
    """Load JSON from GCS.

    Args:
        gcs_path: Full GCS path (gs://bucket/path/file.json).

    Returns:
        Parsed JSON data, or None if not found.
    """
    path = gcs_path.replace(f'gs://{GCS_BUCKET_NAME}/', '')
    bucket = gcs.bucket(GCS_BUCKET_NAME)
    blob = bucket.blob(path)
    try:
        content = blob.download_as_text()
        return json.loads(content)
    except Exception as e:
        print(f'Failed to load {gcs_path}: {e}')
        return None


def gcs_list_blobs(prefix: str) -> list[str]:
    """List blob names under a GCS prefix.

    Args:
        prefix: GCS prefix without bucket (e.g. 'earnings_model/predictions').

    Returns:
        List of blob name strings.
    """
    bucket = gcs.bucket(GCS_BUCKET_NAME)
    return [b.name for b in bucket.list_blobs(prefix=prefix)]


import jquantsapi
jq_cli = jquantsapi.ClientV2(api_key=JQUANTS_API_KEY)

print(f'Setup OK (runtime={RUNTIME})')

## 1. 当日決算取得 + 特徴量生成

ザラバ（`DiscTime < 取引終了時刻`）と引け後を `is_intraday` フラグで区別。


In [ ]:
# ── 1-1. J-Quants: 当日決算サマリー取得 ──
print(f'Fetching fin_summary for {PREDICT_DATE}...')
df_fin = jq_cli.get_fin_summary(date_yyyymmdd=PREDICT_DATE)
print(f'  取得件数: {len(df_fin)}')

if df_fin.empty:
    raise ValueError(f'No fin_summary data for {PREDICT_DATE}.')

# 同日の業績修正を確認
df_rev = df_fin[df_fin['DocType'].str.contains('EarnForecastRevision', na=False)].copy()
rev_tickers: set[str] = set(df_rev['Code'].astype(str).str[:4].tolist()) if not df_rev.empty else set()
print(f'  同日業績修正: {len(df_rev)} 件')

# FinancialStatements のみ（ザラバ + 引け後を is_intraday で区別）
df_fin = df_fin[df_fin['DocType'].str.contains('FinancialStatements', na=False)].copy()

# 訂正報告を除外: CurPerEn が PREDICT_DATE から9ヶ月以上前のものは過去決算の訂正
_predict_dt = pd.Timestamp(PREDICT_DATE_HYPHEN)
_cutoff_dt = _predict_dt - pd.DateOffset(months=9)
df_fin['_cur_per_en'] = pd.to_datetime(df_fin['CurPerEn'], errors='coerce')
_correction_mask = df_fin['_cur_per_en'] < _cutoff_dt
if _correction_mask.any():
    _corr_tickers = df_fin.loc[_correction_mask, 'Code'].astype(str).str[:4].tolist()
    print(f'  訂正報告を除外: {_correction_mask.sum()} 件 ({", ".join(_corr_tickers)})')
df_fin = df_fin[~_correction_mask].copy()
df_fin.drop(columns=['_cur_per_en'], inplace=True)

# 2024-11-05 から東証取引終了が 15:00→15:30 に変更
_close_time = '15:30:00' if PREDICT_DATE >= '20241105' else '15:00:00'
df_fin['is_intraday'] = df_fin['DiscTime'] < _close_time
n_intra = int(df_fin['is_intraday'].sum())
n_after = len(df_fin) - n_intra
print(f'  FinancialStatements: {len(df_fin)} (ザラバ: {n_intra}, 引け後: {n_after})')

# 4桁コード
df_fin['ticker'] = df_fin['Code'].astype(str).str[:4]

# 数値変換
for col in ['OP', 'OdP', 'NP', 'FOP', 'FOdP', 'FNP', 'NxFOP', 'NxFOdP', 'NxFNp', 'FDivAnn', 'ForEPS']:
    if col in df_fin.columns:
        df_fin[col] = pd.to_numeric(df_fin[col], errors='coerce')

# ── 1-3. BQ: 銘柄名 ──
print('Fetching stock names from BQ...')
df_names = bq.query('SELECT TICKER, STOCK_NAME, INDUSTRY_33_CODE, MARKET_CATEGORY FROM `gmailpj-357912.STOCK.STOCK_CODE_LIST`').to_dataframe()
name_map: dict[str, str] = dict(zip(df_names['TICKER'], df_names['STOCK_NAME']))
sector_map: dict[str, str] = dict(zip(df_names['TICKER'], df_names['INDUSTRY_33_CODE'].astype(str)))
market_map: dict[str, str] = dict(zip(df_names['TICKER'], df_names['MARKET_CATEGORY'].fillna('').astype(str)))

# ── 1-4. BQ: 当日株価 + 前日株価（ザラバ答え合わせ用） ──
print('Fetching stock prices from BQ...')
q_price = f"""SELECT TICKER, DATE AS dt, ADJ_CLOSE
FROM `gmailpj-357912.STOCK.STOCK_PRICE_JQUANTS`
WHERE DATE >= DATE_SUB('{PREDICT_DATE_HYPHEN}', INTERVAL 5 DAY)
  AND DATE <= '{PREDICT_DATE_HYPHEN}'
  AND IS_PREFERRED = FALSE
ORDER BY TICKER, DATE DESC"""
df_price = bq.query(q_price).to_dataframe()
# 当日終値
df_price_today = df_price[df_price['dt'].astype(str) == PREDICT_DATE_HYPHEN]
price_map: dict[str, float] = dict(zip(df_price_today['TICKER'], df_price_today['ADJ_CLOSE']))
# 前日終値（ザラバ答え合わせ用）
df_price_prev = df_price[df_price['dt'].astype(str) < PREDICT_DATE_HYPHEN].drop_duplicates('TICKER', keep='first')
prev_price_map: dict[str, float] = dict(zip(df_price_prev['TICKER'], df_price_prev['ADJ_CLOSE']))

# ── 1-5. BQ: コンセンサス（最新1件/銘柄, CURRENT + NEXT） ──
# FY発表時はNEXT（来期）コンセと会社来期予想の乖離を使う（7516コーナンのバグ修正, 2026-04-14）
print('Fetching consensus from BQ...')
q_cons = f"""SELECT TICKER, QUARTER, TARGET, PROFIT AS CONSENSUS_PROFIT FROM (
    SELECT TICKER, QUARTER, TARGET, PROFIT, ROW_NUMBER() OVER (PARTITION BY TICKER, QUARTER, TARGET ORDER BY DATAAT DESC) AS rn
    FROM `gmailpj-357912.STOCK.CONSENSUS`
    WHERE DATAAT <= '{PREDICT_DATE_HYPHEN}' AND TARGET IN ('CURRENT', 'NEXT')
) WHERE rn = 1"""
df_cons = bq.query(q_cons).to_dataframe()
cons_map: dict[tuple[str, str, str], float] = {(r['TICKER'], r['QUARTER'], r['TARGET']): r['CONSENSUS_PROFIT'] for _, r in df_cons.iterrows()}

# ── 1-6. BQ: 前回発表（直近） ──
print('Fetching previous announcements from BQ...')
tickers_sql = ','.join([f"'{t}'" for t in df_fin['ticker'].unique()])
q_prev = f"""SELECT LOCAL_CODE, DISCLOSED_DATE,
    FORECAST_OPERATING_PROFIT, FORECAST_PROFIT,
    OPERATING_PROFIT, TYPE_OF_CURRENT_PERIOD, CURRENT_FISCAL_YEAR_START_DATE
FROM `gmailpj-357912.STOCK.fin_summary`
WHERE SUBSTR(LOCAL_CODE, 1, 4) IN ({tickers_sql})
  AND DISCLOSED_DATE < '{PREDICT_DATE_HYPHEN}'
ORDER BY LOCAL_CODE, DISCLOSED_DATE DESC"""
df_prev = bq.query(q_prev).to_dataframe()

prev_forecast_map: dict[str, dict] = {}
for _, row in df_prev.iterrows():
    tk = str(row['LOCAL_CODE'])[:4]
    if tk not in prev_forecast_map:
        prev_forecast_map[tk] = {'FOP': row['FORECAST_OPERATING_PROFIT'], 'FNP': row['FORECAST_PROFIT']}

# ── 1-6b. 前回配当予想（因子11: 増配/減配） ──
print('Fetching previous dividend forecasts from BQ...')
q_prev_div = f"""SELECT LOCAL_CODE, DISCLOSED_DATE, FORECAST_DIVIDEND_PER_SHARE_ANNUAL
FROM `gmailpj-357912.STOCK.fin_summary`
WHERE SUBSTR(LOCAL_CODE, 1, 4) IN ({tickers_sql})
  AND DISCLOSED_DATE < '{PREDICT_DATE_HYPHEN}'
  AND FORECAST_DIVIDEND_PER_SHARE_ANNUAL IS NOT NULL
  AND FORECAST_DIVIDEND_PER_SHARE_ANNUAL > 0
ORDER BY LOCAL_CODE, DISCLOSED_DATE DESC"""
df_prev_div = bq.query(q_prev_div).to_dataframe()
prev_div_map: dict[str, float] = {}
for _, row in df_prev_div.iterrows():
    tk = str(row['LOCAL_CODE'])[:4]
    if tk not in prev_div_map:
        prev_div_map[tk] = float(row['FORECAST_DIVIDEND_PER_SHARE_ANNUAL'])
print(f'  前回配当予想: {len(prev_div_map)} 銘柄')

# -- 1-7. BQ: Q-on-Q (YoY/QoQ calculation) --
print('Fetching Q-on-Q actuals from BQ...')
q_qoq = f"""SELECT LOCAL_CODE, DISCLOSED_DATE, QUARTER, CURRENT_FISCAL_YEAR_START_DATE, OPERATING_PROFIT, PROFIT
FROM `gmailpj-357912.STOCK.v_fin_summary_actual_for_q_on_q`
WHERE SUBSTR(LOCAL_CODE, 1, 4) IN ({tickers_sql})
  AND DISCLOSED_DATE < '{PREDICT_DATE_HYPHEN}'
ORDER BY LOCAL_CODE, CURRENT_FISCAL_YEAR_START_DATE DESC"""
df_qoq = bq.query(q_qoq).to_dataframe()
df_qoq['CURRENT_FISCAL_YEAR_START_DATE'] = df_qoq['CURRENT_FISCAL_YEAR_START_DATE'].astype(str)

Q_MAP: dict[str, str] = {'1Q': '1Q', '2Q': '2Q', '3Q': '3Q', 'FY': '4Q'}
CUM_PREV_Q: dict[str, str] = {'2Q': '1Q', '3Q': '2Q', 'FY': '3Q'}
PREV_Q_MAP: dict[str, str] = {'2Q': '1Q', '3Q': '2Q', '4Q': '3Q'}

def _date_key(value) -> str:
    _ts = pd.to_datetime(value, errors='coerce')
    return '' if pd.isna(_ts) else _ts.strftime('%Y-%m-%d')

prev_cum_op_map: dict[str, float] = {}
for _, row in df_fin.iterrows():
    tk = row['ticker']
    prev_q_type = CUM_PREV_Q.get(row.get('CurPerType', ''))
    cur_fy_start = _date_key(row.get('CurFYStartDt', row.get('CurFYSt')))
    if not prev_q_type or not cur_fy_start:
        continue
    _matched = df_prev[
        (df_prev['LOCAL_CODE'].str[:4] == tk)
        & (df_prev['TYPE_OF_CURRENT_PERIOD'] == prev_q_type)
        & (df_prev['CURRENT_FISCAL_YEAR_START_DATE'].astype(str) == cur_fy_start)
    ].sort_values('DISCLOSED_DATE', ascending=False)
    if not _matched.empty and pd.notna(_matched.iloc[0]['OPERATING_PROFIT']):
        prev_cum_op_map[tk] = float(_matched.iloc[0]['OPERATING_PROFIT'])

qoq_map: dict[str, dict] = {}
for tk_raw in df_fin['ticker'].unique():
    row0 = df_fin[df_fin['ticker'] == tk_raw].iloc[0]
    cur_per = row0.get('CurPerType', '')
    q_label = Q_MAP.get(cur_per, '')
    cur_fy_start = _date_key(row0.get('CurFYStartDt', row0.get('CurFYSt')))
    jq_op_cum = row0.get('OP')

    cur_standalone_op = None
    if pd.notna(jq_op_cum):
        if cur_per == '1Q':
            cur_standalone_op = float(jq_op_cum)
        else:
            prev_cum = prev_cum_op_map.get(tk_raw)
            cur_standalone_op = float(jq_op_cum) - prev_cum if prev_cum is not None else float(jq_op_cum)

    tk_qoq = df_qoq[(df_qoq['LOCAL_CODE'].str[:4] == tk_raw) & (df_qoq['QUARTER'] == q_label)]
    if cur_fy_start:
        tk_qoq = tk_qoq[tk_qoq['CURRENT_FISCAL_YEAR_START_DATE'].astype(str) < cur_fy_start]
    tk_qoq = tk_qoq.sort_values('CURRENT_FISCAL_YEAR_START_DATE', ascending=False)

    prev_year_op = tk_qoq.iloc[0]['OPERATING_PROFIT'] if len(tk_qoq) >= 1 else None
    yoy_op_val = None
    if cur_standalone_op is not None and prev_year_op is not None and pd.notna(prev_year_op) and prev_year_op != 0:
        yoy_op_val = float((cur_standalone_op - float(prev_year_op)) / abs(float(prev_year_op)))

    qoq_op_val = None
    prev_q_label = PREV_Q_MAP.get(q_label)
    if prev_q_label and cur_standalone_op is not None and cur_fy_start:
        _prev_q = df_qoq[
            (df_qoq['LOCAL_CODE'].str[:4] == tk_raw)
            & (df_qoq['QUARTER'] == prev_q_label)
            & (df_qoq['CURRENT_FISCAL_YEAR_START_DATE'].astype(str) == cur_fy_start)
        ].sort_values('DISCLOSED_DATE', ascending=False)
        if not _prev_q.empty:
            _prev_q_op = _prev_q.iloc[0]['OPERATING_PROFIT']
            if pd.notna(_prev_q_op) and _prev_q_op != 0:
                qoq_op_val = float((cur_standalone_op - float(_prev_q_op)) / abs(float(_prev_q_op)))

    qoq_map[tk_raw] = {'yoy_op': yoy_op_val, 'qoq_op': qoq_op_val}

# ── 1-7b. BQ: 過去FY YoY OP成長率（因子7: 成長減速/加速） ──
print('Fetching historical FY YoY OP growth from BQ...')
q_fy_yoy = f"""SELECT
    SUBSTR(LOCAL_CODE, 1, 4) AS ticker,
    CURRENT_FISCAL_YEAR_START_DATE AS fy_start,
    OPERATING_PROFIT
FROM `gmailpj-357912.STOCK.v_fin_summary_actual_for_q_on_q`
WHERE SUBSTR(LOCAL_CODE, 1, 4) IN ({tickers_sql})
  AND QUARTER = '4Q'
  AND DISCLOSED_DATE < '{PREDICT_DATE_HYPHEN}'
ORDER BY LOCAL_CODE, CURRENT_FISCAL_YEAR_START_DATE DESC"""
df_fy_yoy = bq.query(q_fy_yoy).to_dataframe()

baseline_yoy_op_map: dict[str, float] = {}
for tk_raw in df_fin['ticker'].unique():
    tk_fy = df_fy_yoy[df_fy_yoy['ticker'] == tk_raw].sort_values('fy_start', ascending=False)
    if len(tk_fy) >= 3:  # 少なくとも3年分（2期分のYoY）
        ops = tk_fy['OPERATING_PROFIT'].tolist()
        yoy_list: list[float] = []
        for j in range(len(ops) - 1):
            cur, prev = ops[j], ops[j + 1]
            if pd.notna(cur) and pd.notna(prev) and prev != 0:
                yoy_list.append(float((cur - prev) / abs(prev)))
        if yoy_list:
            baseline_yoy_op_map[tk_raw] = float(pd.Series(yoy_list).median())
print(f'  Baseline YoY OP: {len(baseline_yoy_op_map)} 銘柄')

# ── 1-7c. TDnet開示タイトル検索（因子8: 記念配当、因子10: 自社株買い） ──
# yanoshin JSON API → TDnet HTML → BQ（フォールバック3段）
_today_str = datetime.now(tz=JST).strftime('%Y%m%d')
special_div_tickers: set[str] = set()
buyback_tickers: dict[str, float] = {}  # ticker → 発行済比率（タイトルからは不明なので0）
_target_tickers: set[str] = set(df_fin['ticker'].unique())

_BUYBACK_KW = ['自己株式の取得', '自社株買い']
_SPECIAL_DIV_KW = ['記念配当', '特別配当']

def _fetch_tdnet_yanoshin(date_str: str) -> list[dict]:
    """yanoshin JSON API から当日全開示を取得."""
    import httpx
    url = f'https://webapi.yanoshin.jp/webapi/tdnet/list/{date_str}-{date_str}.json?limit=9999'
    try:
        resp = httpx.get(url, timeout=30)
        resp.raise_for_status()
        items = resp.json().get('items', [])
        return [{'code': (t.get('company_code') or '')[:4], 'title': t.get('title', '')}
                for item in items if (t := item.get('Tdnet'))]
    except Exception as e:
        print(f'  yanoshin fetch failed: {e}')
        return []

def _fetch_tdnet_html(date_str: str) -> list[dict]:
    """TDnet HTML スクレイピングでフォールバック取得."""
    import httpx
    from bs4 import BeautifulSoup
    results = []
    _page = 1
    while True:
        _url = f'https://www.release.tdnet.info/inbs/I_list_{_page:03d}_{date_str}.html'
        try:
            _resp = httpx.get(_url, timeout=15, verify=False)
        except Exception:
            break
        if _resp.status_code == 404:
            break
        _resp.encoding = 'utf-8'
        _soup = BeautifulSoup(_resp.text, 'lxml')
        _rows_found = 0
        for _tr in _soup.find_all('tr'):
            _cells = _tr.find_all('td')
            if len(_cells) < 4:
                continue
            if 'kjTime' not in ' '.join(_cells[0].get('class', [])):
                continue
            _rows_found += 1
            results.append({'code': _cells[1].get_text(strip=True)[:4], 'title': _cells[3].get_text(strip=True)})
        if _rows_found == 0:
            break
        _page += 1
        time.sleep(0.5)
    return results

if PREDICT_DATE >= _today_str:
    # 当日: yanoshin API → TDnet HTML フォールバック
    print('Fetching TDnet disclosures (yanoshin API)...')
    _disclosures = _fetch_tdnet_yanoshin(PREDICT_DATE)
    if not _disclosures:
        print('  yanoshin empty, falling back to TDnet HTML...')
        _disclosures = _fetch_tdnet_html(PREDICT_DATE)
    for d in _disclosures:
        _code, _title = d['code'], d['title']
        if _code not in _target_tickers:
            continue
        if any(kw in _title for kw in _SPECIAL_DIV_KW):
            special_div_tickers.add(_code)
            print(f'  記念配当/特別配当: {_code} | {_title}')
        if any(kw in _title for kw in _BUYBACK_KW):
            buyback_tickers[_code] = 0.0
            print(f'  自社株買い: {_code} | {_title}')
else:
    # 過去: BQ TDNET_DOCUMENTS_ENHANCED
    print('Fetching TDnet disclosures from BQ...')
    _tickers_sql = ','.join([f"'{t}'" for t in _target_tickers])
    _q_tdnet = f"""SELECT DISTINCT TICKER, DOC_TITLE
    FROM `gmailpj-357912.STOCK.TDNET_DOCUMENTS_ENHANCED`
    WHERE SUBMISSION_DATE = '{PREDICT_DATE_HYPHEN}'
      AND TICKER IN ({_tickers_sql})
      AND (DOC_TITLE LIKE '%記念配当%' OR DOC_TITLE LIKE '%特別配当%'
           OR DOC_TITLE LIKE '%自己株式の取得%' OR DOC_TITLE LIKE '%自社株買い%')"""
    _df_tdnet = bq.query(_q_tdnet).to_dataframe()
    if not _df_tdnet.empty:
        for _, _r in _df_tdnet.iterrows():
            _tk, _ttl = _r['TICKER'], _r['DOC_TITLE']
            if any(kw in _ttl for kw in _SPECIAL_DIV_KW):
                special_div_tickers.add(_tk)
            if any(kw in _ttl for kw in _BUYBACK_KW):
                buyback_tickers[_tk] = 0.0
print(f'  記念配当/特別配当: {len(special_div_tickers)} 銘柄, 自社株買い: {len(buyback_tickers)} 銘柄')

# ── 1-7d. GCS: 20日β読み込み（因子9: テーマブースト） ──
print('Loading 20-day beta from GCS...')
_beta_blob = f'earnings_model/beta_20d.csv'
try:
    _beta_csv = gcs.bucket(GCS_BUCKET_NAME).blob(_beta_blob).download_as_text()
    import io as _io
    df_beta = pd.read_csv(_io.StringIO(_beta_csv))
    beta_map: dict[str, float] = dict(zip(df_beta['TICKER'], df_beta['beta_20d']))
    print(f'  β: {len(beta_map)} 銘柄 (calc_date={df_beta["calc_date"].iloc[0]})')
except Exception as _e:
    print(f'  β読み込み失敗（スキップ）: {_e}')
    beta_map = {}

# ── 1-7e. TOPIX当日リターン取得 ──
print('Fetching TOPIX return for predict date...')
_q_topix = f"""SELECT
  SAFE_DIVIDE(t1.CLOSE - t0.CLOSE, t0.CLOSE) AS topix_ret
FROM `gmailpj-357912.STOCK.INDEX_PRICE` t1
JOIN (
  SELECT DATE, CLOSE FROM `gmailpj-357912.STOCK.INDEX_PRICE`
  WHERE INDEX_CODE = '0000' AND DATE < '{PREDICT_DATE_HYPHEN}'
  ORDER BY DATE DESC LIMIT 1
) t0 ON TRUE
WHERE t1.INDEX_CODE = '0000' AND t1.DATE = '{PREDICT_DATE_HYPHEN}'
"""
_df_topix = bq.query(_q_topix).to_dataframe()
topix_ret: float = float(_df_topix['topix_ret'].iloc[0]) if not _df_topix.empty else 0.0
print(f'  TOPIX当日リターン: {topix_ret:+.2%}')

# ── 1-8. 特徴量計算 ──
print('Computing features...')
results: list[dict] = []
for _, row in df_fin.iterrows():
    tk: str = row['ticker']
    cur_per: str = row.get('CurPerType', '')
    op = row.get('OP')
    fop = row.get('FOP')
    odp = row.get('OdP')
    nx_fop = row.get('NxFOP')
    nx_fodp = row.get('NxFOdP')

    # progress_op
    progress_op: float | None = None
    if pd.notna(op) and pd.notna(fop) and fop != 0 and cur_per in ('1Q', '2Q', '3Q'):
        progress_op = float(op / fop)

    # guidance revision
    prev = prev_forecast_map.get(tk, {})
    prev_fop = prev.get('FOP')
    has_guidance_revision: bool = False
    guidance_op_change: float | None = None
    if pd.notna(fop) and pd.notna(prev_fop) and prev_fop != 0:
        change = (fop - prev_fop) / abs(prev_fop)
        if abs(change) > 0.001:
            has_guidance_revision = True
            guidance_op_change = float(change)

    # yoy_op
    yoy_data = qoq_map.get(tk, {})
    yoy_op: float | None = yoy_data.get('yoy_op')

    # consensus deviation (PROFIT is in million yen)
    # FY: 来期会社予想 vs NEXT コンセ（市場反応の主因は来期ガイダンス）
    # 1Q/2Q/3Q: 今期累積実績 vs CURRENT コンセ
    consensus_deviation: float | None = None
    _f4_source: str = ''  # コンセ比較のソース（FY_NEXT / FY_CURRENT_fallback / Q_CURRENT）
    if cur_per == 'FY':
        cons_next = cons_map.get((tk, 'FY', 'NEXT'))
        if pd.notna(nx_fodp) and cons_next is not None and cons_next != 0:
            cons_next_yen = cons_next * 1_000_000
            consensus_deviation = float((nx_fodp - cons_next_yen) / abs(cons_next_yen))
            _f4_source = 'FY_NEXT'
        else:
            # フォールバック: 来期予想未開示なら今期実績 vs CURRENT コンセで比較（2026-04-16追加, 2379ディップ起点）
            cons_cur = cons_map.get((tk, 'FY', 'CURRENT'))
            if pd.notna(odp) and cons_cur is not None and cons_cur != 0:
                cons_cur_yen = cons_cur * 1_000_000
                consensus_deviation = float((odp - cons_cur_yen) / abs(cons_cur_yen))
                _f4_source = 'FY_CURRENT_fallback'
    else:
        cons_profit = cons_map.get((tk, cur_per, 'CURRENT'))
        if pd.notna(odp) and cons_profit is not None and cons_profit != 0:
            cons_yen = cons_profit * 1_000_000
            consensus_deviation = float((odp - cons_yen) / abs(cons_yen))
            _f4_source = 'Q_CURRENT'

    # next year OP change (FY only)
    next_year_op_change: float | None = None
    if cur_per == 'FY' and pd.notna(nx_fop) and pd.notna(op) and op != 0:
        next_year_op_change = float((nx_fop - op) / abs(op))

    # selloff risk
    selloff_risk: bool = False
    if cur_per == '3Q' and progress_op is not None and progress_op > 0.90 and not has_guidance_revision:
        selloff_risk = True

    # beta & topix
    _beta = beta_map.get(tk)
    _theme_boost = False
    if _beta is not None and _beta > 1.0 and topix_ret > 0:
        _theme_boost = True

    results.append({
        'ticker': tk,
        'name': name_map.get(tk, ''),
        'industry_33': sector_map.get(tk, ''),
        'market_division': market_map.get(tk, ''),
        'quarter': cur_per,
        'is_intraday': bool(row['is_intraday']),
        'disc_time': str(row.get('DiscTime', '')),
        'op': op,
        'fop': fop,
        'odp': odp,
        'adj_close': price_map.get(tk),
        'prev_close': prev_price_map.get(tk),
        'progress_op': progress_op,
        'has_guidance_revision': has_guidance_revision,
        'guidance_op_change': guidance_op_change,
        'yoy_op': yoy_op,
        'consensus_deviation': consensus_deviation,
        'f4_source': _f4_source if _f4_source else None,
        'next_year_op_change': next_year_op_change,
        'next_year_disclosed': bool(cur_per == 'FY' and pd.notna(nx_fop)),
        'selloff_risk': selloff_risk,
        'baseline_yoy_op': baseline_yoy_op_map.get(tk),
        'has_special_dividend': tk in special_div_tickers,
        'has_buyback': tk in buyback_tickers,
        'div_change': float((float(row.get('FDivAnn', 0)) - prev_div_map[tk]) / prev_div_map[tk])
            if pd.notna(row.get('FDivAnn')) and tk in prev_div_map and prev_div_map[tk] > 0
            else None,
        'qoq_op': qoq_map.get(tk, {}).get('qoq_op'),
        'per': float(price_map.get(tk, float('nan')) / float(row.get('ForEPS', 0)))
            if price_map.get(tk) and pd.notna(row.get('ForEPS')) and float(row.get('ForEPS', 0)) > 0
            else None,
        'beta_20d': _beta,
        'theme_boost': _theme_boost,
    })

df_features = pd.DataFrame(results)
print(f'\n特徴量計算完了: {len(df_features)} 銘柄 (ザラバ: {df_features["is_intraday"].sum()}, 引け後: {(~df_features["is_intraday"]).sum()})')
df_features


## 2. スコアリング & 予測

In [ ]:
# ── 2-1. スコアリング ──
def compute_score(row: pd.Series) -> tuple[int, list[str]]:
    """Compute prediction score and reasons for a single stock.

    Args:
        row: Feature row from df_features.

    Returns:
        Tuple of (score, list of reason strings).
    """
    score: int = 0
    reasons: list[str] = []
    cur_per: str = row['quarter']
    exp: float | None = {'1Q': 0.25, '2Q': 0.50, '3Q': 0.75}.get(cur_per)

    # Factor 1: Progress rate
    prog = row.get('progress_op')
    if prog is not None and exp is not None:
        if prog > exp * 1.2:
            score += 1
            reasons.append(f'進捗率高 {prog:.0%} (期待{exp:.0%})')
        elif prog < exp * 0.8:
            score -= 1
            reasons.append(f'進捗率低 {prog:.0%} (期待{exp:.0%})')

    # Factor 2: Guidance revision
    gc = row.get('guidance_op_change')
    if row.get('has_guidance_revision') and gc is not None:
        if gc > 0:
            score += 1
            reasons.append(f'上方修正 {gc:+.1%}')
        elif gc < 0:
            score -= 1
            reasons.append(f'下方修正 {gc:+.1%}')

    # Factor 3: YoY OP
    yoy = row.get('yoy_op')
    if yoy is not None:
        if yoy > 0.30:
            score += 1
            reasons.append(f'YoY OP +{yoy:.0%}')
        elif yoy < -0.30:
            score -= 1
            reasons.append(f'YoY OP {yoy:.0%}')

    # Factor 4: Consensus deviation（段階的スコア）
    cd = row.get('consensus_deviation')
    if cd is not None and pd.notna(cd):
        if cd > 0.10:
            score += 3
            reasons.append(f'コンセ乖離 {cd:+.1%}')
        elif cd > 0.05:
            score += 2
            reasons.append(f'コンセ乖離 {cd:+.1%}')
        elif cd > 0:
            score += 1
            reasons.append(f'コンセ乖離 {cd:+.1%}')
        elif cd < -0.10:
            score -= 3
            reasons.append(f'コンセ乖離 {cd:+.1%}')
        elif cd < -0.05:
            score -= 2
            reasons.append(f'コンセ乖離 {cd:+.1%}')
        elif cd < 0:
            score -= 1
            reasons.append(f'コンセ乖離 {cd:+.1%}')

    # Factor 5: Next year guidance (FY only)
    nyc = row.get('next_year_op_change')
    if nyc is not None and pd.notna(nyc):
        if nyc > 0.10:
            score += 2
            reasons.append(f'来期OP増益 {nyc:+.1%}')
        elif nyc < -0.10:
            score -= 2
            reasons.append(f'来期OP減益 {nyc:+.1%}')
    elif cur_per == 'FY' and row.get('next_year_disclosed') is False:
        reasons.append('来期予想未開示 (F5/F7/F12無効)')

    # Factor 6: Selloff risk
    if row.get('selloff_risk'):
        score -= 2
        reasons.append('売り圧力リスク(3Q高進捗+修正なし)')

    # Factor 7: Growth deceleration/acceleration (FY only)
    nyc7 = row.get('next_year_op_change')
    baseline = row.get('baseline_yoy_op')
    if cur_per == 'FY' and nyc7 is not None and pd.notna(nyc7) and baseline is not None and pd.notna(baseline):
        gap = nyc7 - baseline
        if gap > 0.20:
            score += 1
            reasons.append(f'成長加速 (翌期{nyc7:+.0%} vs baseline{baseline:+.0%})')
        elif gap < -0.20:
            score -= 1
            reasons.append(f'成長減速 (翌期{nyc7:+.0%} vs baseline{baseline:+.0%})')

    # Factor 8: Special/memorial dividend
    if row.get('has_special_dividend'):
        score += 1
        reasons.append('記念配当/特別配当')

    # Factor 9: Theme boost (high beta × risk-on)
    if row.get('theme_boost'):
        score += 1
        _b = row.get('beta_20d', 0)
        reasons.append(f'テーマブースト (β={_b:.1f}, TOPIX+)')

    # Factor 10: Buyback (自社株買い)
    if row.get('has_buyback'):
        score += 2
        reasons.append('自社株買い')

    # Factor 11: Dividend change (増配/減配, J-Quants fin_summary ベース)
    div_chg = row.get('div_change')
    if div_chg is not None and pd.notna(div_chg):
        if div_chg > 0.05:
            score += 1
            reasons.append(f'増配 {div_chg:+.0%}')
        elif div_chg < -0.20:
            score -= 3
            reasons.append(f'大幅減配 {div_chg:+.0%}')
        elif div_chg < -0.05:
            score -= 2
            reasons.append(f'減配 {div_chg:+.0%}')

    # Factor 12: PER valuation (PEG-based, FY only)
    _per = row.get('per')
    _nyc12 = row.get('next_year_op_change')
    if _per is not None and pd.notna(_per) and _per > 0 and _nyc12 is not None and pd.notna(_nyc12) and _nyc12 > 0:
        _growth_pct = _nyc12 * 100  # 0.20 → 20
        _peg = _per / _growth_pct if _growth_pct > 0 else float('inf')
        if _peg < 0.5:
            score += 2
            reasons.append(f'PEG割安 {_peg:.1f} (PER{_per:.0f}x/成長{_growth_pct:.0f}%)')
        elif _peg < 1.0:
            score += 1
            reasons.append(f'PEG割安 {_peg:.1f} (PER{_per:.0f}x/成長{_growth_pct:.0f}%)')
        elif _peg > 2.0:
            score -= 1
            reasons.append(f'PEG割高 {_peg:.1f} (PER{_per:.0f}x/成長{_growth_pct:.0f}%)')

    # Factor 13: QoQ OP change (単独四半期の前Q比急変)
    _qoq = row.get('qoq_op')
    if _qoq is not None and pd.notna(_qoq):
        if _qoq > 0.50:
            score += 1
            reasons.append(f'QoQ OP急伸 {_qoq:+.0%}')
        elif _qoq < -0.50:
            score -= 2
            reasons.append(f'QoQ OP急落 {_qoq:+.0%}')

    return score, reasons


def score_to_prediction(score: int) -> str:
    """Map score to prediction category.

    Args:
        score: Integer score from compute_score.

    Returns:
        Prediction category string.
    """
    if score >= 2:
        return 'UP'
    elif score <= -2:
        return 'DOWN'
    else:
        return 'NEUTRAL'


# Apply scoring
scores: list[int] = []
predictions: list[str] = []
all_reasons: list[str] = []

for _, row in df_features.iterrows():
    s, r = compute_score(row)
    scores.append(s)
    predictions.append(score_to_prediction(s))
    all_reasons.append(' / '.join(r) if r else 'シグナルなし')

df_features['score'] = scores
df_features['prediction'] = predictions
df_features['reasons'] = all_reasons

# ── 2-2. 結果表示 ──
display_cols = ['ticker', 'name', 'quarter', 'is_intraday', 'score', 'prediction', 'reasons']
print(f'\n=== 予測結果 ({PREDICT_DATE_HYPHEN}) ===')
print(f'対象銘柄数: {len(df_features)} (ザラバ: {df_features["is_intraday"].sum()}, 引け後: {(~df_features["is_intraday"]).sum()})')
print()

# Score distribution
print('スコア分布:')
print(df_features['prediction'].value_counts().to_string())
print()

# Display table
df_display = df_features[display_cols].sort_values('score', ascending=False)
display(df_display)

# ── 2-3. GCS に保存 ──
prediction_records = df_features[
    ['ticker', 'name', 'industry_33', 'market_division', 'quarter', 'is_intraday', 'disc_time',
     'score', 'prediction', 'reasons',
     'progress_op', 'has_guidance_revision', 'guidance_op_change',
     'yoy_op', 'consensus_deviation', 'f4_source', 'next_year_op_change', 'next_year_disclosed', 'selloff_risk',
     'baseline_yoy_op', 'has_special_dividend',
     'has_buyback', 'div_change', 'qoq_op', 'per',
     'adj_close', 'prev_close']
].to_dict(orient='records')

prediction_payload: dict = {
    'predict_date': PREDICT_DATE,
    'created_at': datetime.now(tz=JST).isoformat(),
    'count': len(prediction_records),
    'predictions': prediction_records,
}

_ts = datetime.now(tz=JST).strftime('%H%M%S')
gcs_path = f'{GCS_PREDICTIONS}/prediction_{PREDICT_DATE}_{_ts}.json'
gcs_save_json(prediction_payload, gcs_path)
print(f'\n予測を GCS に保存しました: {gcs_path}')


## 3. 答え合わせ

- **ザラバ銘柄** (`is_intraday=True`): 当日終値 vs 前日終値（Step 2 の株価データで即時計算）
- **引け後銘柄** (`is_intraday=False`): 翌日(`ACTUAL_DATE`)終値 vs 当日終値


In [ ]:
# ── 初期化ガード（Step 3 単独実行対応） ──
if 'gcs_list_blobs' not in dir():
    print('初期化セルが未実行のため自動実行します...')
    # Cell[1]: 日付設定 — 未定義なら実行を促す
    if 'PREDICT_DATE' not in dir():
        raise RuntimeError(
            '★ 日付設定セル（PREDICT_DATE / ACTUAL_DATE）を先に実行してください'
        )
    # Cell[2]: pip install
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'jquants-api-client'], capture_output=True)
    # Cell[3]: 初期化を再実行
    # --- imports ---
    import os, json, time
    from pathlib import Path
    from datetime import datetime, timedelta, date
    import pandas as pd
    import numpy as np
    import matplotlib.pyplot as plt
    import matplotlib
    import warnings
    warnings.filterwarnings('ignore')
    try:
        from google.colab import auth, userdata
        RUNTIME = 'colab'
    except ImportError:
        RUNTIME = 'local'
    if RUNTIME == 'colab':
        auth.authenticate_user()
        from google.cloud import bigquery, storage
        bq = bigquery.Client(project='gmailpj-357912')
        gcs = storage.Client(project='gmailpj-357912')
    else:
        from google.cloud import bigquery, storage
        from google.oauth2 import service_account
        PROJECT_ROOT = Path(r'C:\gdrive\claude\investment-agent')
        sys.path.insert(0, str(PROJECT_ROOT / 'scripts'))
        from dotenv import load_dotenv
        load_dotenv(PROJECT_ROOT / '.env')
        KEY_FILE = str(PROJECT_ROOT / 'keys' / 'gcp-service-account.json')
        creds = service_account.Credentials.from_service_account_file(KEY_FILE)
        bq = bigquery.Client(credentials=creds, project='gmailpj-357912')
        gcs = storage.Client(credentials=creds, project='gmailpj-357912')
    GCS_BUCKET = 'gs://stock_data_1930932'
    GCS_BUCKET_NAME = 'stock_data_1930932'
    GCS_PREDICTIONS = f'{GCS_BUCKET}/earnings_model/predictions'
    GCS_ACTUALS = f'{GCS_BUCKET}/earnings_model/actuals'
    GCS_ACCURACY = f'{GCS_BUCKET}/earnings_model/accuracy'
    def gcs_save_json(data, gcs_path):
        path = gcs_path.replace(f'gs://{GCS_BUCKET_NAME}/', '')
        bucket = gcs.bucket(GCS_BUCKET_NAME)
        blob = bucket.blob(path)
        blob.upload_from_string(json.dumps(data, ensure_ascii=False, indent=2, default=str), content_type='application/json')
        print(f'Saved to {gcs_path}')
    def gcs_load_json(gcs_path):
        path = gcs_path.replace(f'gs://{GCS_BUCKET_NAME}/', '')
        bucket = gcs.bucket(GCS_BUCKET_NAME)
        blob = bucket.blob(path)
        try:
            return json.loads(blob.download_as_text())
        except Exception as e:
            print(f'Failed to load {gcs_path}: {e}'); return None
    def gcs_list_blobs(prefix):
        bucket = gcs.bucket(GCS_BUCKET_NAME)
        return [b.name for b in bucket.list_blobs(prefix=prefix)]
    print(f'初期化完了 (runtime={RUNTIME})')

# ── 3-1. 予測ロード（同日の最新ファイルを自動選択） ──
_pred_blobs = [b for b in gcs_list_blobs(f'earnings_model/predictions/prediction_{PREDICT_DATE}_') if b.endswith('.json')]
if not _pred_blobs:
    raise ValueError(f'Prediction file not found for {PREDICT_DATE}')
_pred_blob = sorted(_pred_blobs)[-1]  # タイムスタンプ降順で最新
pred_path = f'gs://{GCS_BUCKET_NAME}/{_pred_blob}'
pred_data = gcs_load_json(pred_path)
if pred_data is None:
    raise ValueError(f'Failed to load: {pred_path}')
df_pred = pd.DataFrame(pred_data['predictions'])
print(f'予測ロード: {len(df_pred)} 銘柄 (file={_pred_blob})')

# is_intraday が含まれない旧データの後方互換
if 'is_intraday' not in df_pred.columns:
    df_pred['is_intraday'] = False

n_intra = int(df_pred['is_intraday'].sum())
n_after = len(df_pred) - n_intra
print(f'  ザラバ: {n_intra}, 引け後: {n_after}')

# ── 3-2a. Guard: ACTUAL_DATE の株価データ投入済みチェック（2026-04-16 追加） ──
# 引け後銘柄がある場合のみチェック。翌営業日引け後データ投入前の誤実行を未然検知
_n_after = int((~df_pred['is_intraday']).sum()) if 'is_intraday' in df_pred.columns else len(df_pred)
if PREDICT_DATE == ACTUAL_DATE:
    raise RuntimeError(
        f'★ PREDICT_DATE({PREDICT_DATE}) == ACTUAL_DATE({ACTUAL_DATE})。'
        f'ACTUAL_DATE は翌営業日に設定してください'
    )
if _n_after > 0:
    _q_check = f"SELECT COUNT(*) AS cnt FROM `gmailpj-357912.STOCK.STOCK_PRICE_JQUANTS` WHERE DATE = '{ACTUAL_DATE_HYPHEN}'"
    _n_rows = int(bq.query(_q_check).to_dataframe().iloc[0]['cnt'])
    if _n_rows < 3000:  # 全銘柄約4000。3000未満なら未投入
        raise RuntimeError(
            f'★ ACTUAL_DATE={ACTUAL_DATE_HYPHEN} の STOCK_PRICE_JQUANTS 投入不完全 ({_n_rows}行)。'
            f'翌営業日引け後に株価データ投入完了後 (通常 18:00 JST 以降) に再実行してください'
        )
    print(f'Guard OK: STOCK_PRICE_JQUANTS on {ACTUAL_DATE_HYPHEN} = {_n_rows} rows')

# ── 3-2. 実績株価取得 ──
tickers_sql = ','.join([f"'{t}'" for t in df_pred['ticker'].unique()])

# ザラバ銘柄: 当日終値 vs 前日終値 → prev_close は prediction に含まれている
# 引け後銘柄: 翌日(ACTUAL_DATE)終値 vs 当日終値
print(f'Fetching actual prices for {ACTUAL_DATE_HYPHEN} (引け後銘柄用)...')
q_actual = f"""
SELECT
    a.TICKER,
    a.ADJ_CLOSE AS actual_close,
    p.ADJ_CLOSE AS prev_close
FROM `gmailpj-357912.STOCK.STOCK_PRICE_JQUANTS` a
JOIN `gmailpj-357912.STOCK.STOCK_PRICE_JQUANTS` p
  ON a.TICKER = p.TICKER AND p.DATE = '{PREDICT_DATE_HYPHEN}'
WHERE a.DATE = '{ACTUAL_DATE_HYPHEN}'
  AND a.IS_PREFERRED = FALSE
  AND p.IS_PREFERRED = FALSE
  AND a.TICKER IN ({tickers_sql})
"""
df_actual_after = bq.query(q_actual).to_dataframe()
print(f'  引け後用株価取得: {len(df_actual_after)} 銘柄')

# ザラバ銘柄: 当日終値 vs 前日終値（prediction に保存済みの adj_close / prev_close を使用）
print('Computing intraday returns from prediction data...')
df_intra = df_pred[df_pred['is_intraday']].copy()
df_intra['actual_close'] = pd.to_numeric(df_intra['adj_close'], errors='coerce')
df_intra['prev_close_val'] = pd.to_numeric(df_intra['prev_close'], errors='coerce')
df_intra['actual_return'] = (
    (df_intra['actual_close'] - df_intra['prev_close_val']) / df_intra['prev_close_val']
)
df_intra_result = df_intra[['ticker', 'actual_return', 'actual_close']].copy()
df_intra_result['prev_close'] = df_intra['prev_close_val']
df_intra_result.rename(columns={'ticker': 'TICKER'}, inplace=True)
print(f'  ザラバ答え合わせ: {df_intra_result["actual_return"].notna().sum()} 銘柄')

# 引け後銘柄: 翌日終値を使用
df_after = df_pred[~df_pred['is_intraday']].copy()
df_actual_after_merged = df_after[['ticker']].merge(
    df_actual_after[['TICKER', 'actual_close', 'prev_close']].assign(
        actual_return=lambda x: (x['actual_close'] - x['prev_close']) / x['prev_close']
    ),
    left_on='ticker', right_on='TICKER', how='left'
)

# 統合
df_actual_all = pd.concat([df_intra_result, df_actual_after_merged[['TICKER', 'actual_return', 'actual_close', 'prev_close']]], ignore_index=True)


# ── 3-3. リターン分類 ──
def classify_return(ret: float) -> str:
    """Classify actual return into prediction categories.

    Args:
        ret: Actual return as a decimal (e.g., 0.02 = +2%).

    Returns:
        Category string matching prediction labels.
    """
    if ret > 0.02:
        return 'UP'
    elif ret < -0.02:
        return 'DOWN'
    else:
        return 'NEUTRAL'


df_actual_all['actual_category'] = df_actual_all['actual_return'].apply(
    lambda x: classify_return(x) if pd.notna(x) else None
)

# ── 3-4. 突合 ──
df_compare = df_pred.merge(
    df_actual_all[['TICKER', 'actual_return', 'actual_category', 'actual_close', 'prev_close']],
    left_on='ticker', right_on='TICKER', how='left',
    suffixes=('', '_actual')
)

# 方向一致判定
direction_map: dict[str, int] = {
    'UP': 1, 'NEUTRAL': 0, 'DOWN': -1
}
df_compare['pred_dir'] = df_compare['prediction'].map(direction_map)
df_compare['actual_dir'] = df_compare['actual_category'].map(direction_map)
df_compare['direction_match'] = df_compare['pred_dir'] == df_compare['actual_dir']

# 結果表示
matched = df_compare['actual_return'].notna()
n_matched: int = int(matched.sum())
n_intra_matched = int((df_compare.loc[matched, 'is_intraday']).sum())
n_after_matched = n_matched - n_intra_matched

print(f'\n=== 答え合わせ ({PREDICT_DATE_HYPHEN}) ===')
print(f'突合成功: {n_matched} / {len(df_compare)} 銘柄 (ザラバ: {n_intra_matched}, 引け後: {n_after_matched})')

dir_acc: float = 0.0
score_ret_corr: float = 0.0
if n_matched > 0:
    dir_acc = float(df_compare.loc[matched, 'direction_match'].mean())
    score_ret_corr = float(
        df_compare.loc[matched, ['score', 'actual_return']].corr().iloc[0, 1]
    )
    print(f'方向一致率: {dir_acc:.1%}')
    print(f'スコア x リターン相関: {score_ret_corr:.3f}')

    # ザラバ / 引け後別の精度
    for label, mask in [('ザラバ', df_compare['is_intraday']), ('引け後', ~df_compare['is_intraday'])]:
        sub = df_compare[matched & mask]
        if len(sub) > 0:
            sub_acc = float(sub['direction_match'].mean())
            sub_corr = float(sub[['score', 'actual_return']].corr().iloc[0, 1]) if len(sub) > 1 else 0.0
            print(f'  {label}: {len(sub)}銘柄, 方向一致率 {sub_acc:.1%}, 相関 {sub_corr:.3f}')

    print()
    display_cols = ['ticker', 'name', 'quarter', 'is_intraday', 'score', 'prediction',
                    'actual_return', 'actual_category', 'direction_match']
    df_show = df_compare.loc[matched, display_cols].sort_values('score', ascending=False).copy()
    df_show['actual_return'] = df_show['actual_return'].apply(lambda x: f'{x:+.2%}')
    display(df_show)

# ── 3-5. GCS に保存 ──
# 比較前終値 / 比較後終値 を含めて保存（2026-04-15 追加）
# ザラバ: before=発表日前日終値, after=発表日終値
# 引け後: before=発表日終値, after=翌日終値
actual_records = df_compare[
    ['ticker', 'name', 'quarter', 'is_intraday', 'score', 'prediction',
     'actual_return', 'actual_category', 'direction_match',
     'prev_close_actual', 'actual_close']
].rename(columns={
    'prev_close_actual': 'compare_before_close',
    'actual_close': 'compare_after_close',
}).to_dict(orient='records')

actual_payload: dict = {
    'predict_date': PREDICT_DATE,
    'actual_date': ACTUAL_DATE,
    'created_at': datetime.now(tz=JST).isoformat(),
    'count': len(actual_records),
    'direction_accuracy': dir_acc if n_matched > 0 else None,
    'score_return_correlation': score_ret_corr if n_matched > 0 else None,
    'actuals': actual_records,
}
# 保存日 + 時刻 + 予測対象日 の形式（2026-04-15 変更）
_now_jst = datetime.now(tz=JST)
_save_date = _now_jst.strftime('%Y%m%d')
_ts_a = _now_jst.strftime('%H%M%S')
actual_gcs_path = f'{GCS_ACTUALS}/actual_{_save_date}_{_ts_a}_for_{PREDICT_DATE}.json'
gcs_save_json(actual_payload, actual_gcs_path)
print(f'\n実績を GCS に保存しました: {actual_gcs_path}')


## 4. 精度集計

In [ ]:
# ── 4-1. 全期間の予測/実績ファイルをロード ──
print('Loading all prediction/actual files from GCS...')
actual_blobs = gcs_list_blobs('earnings_model/actuals/')
actual_blobs = [b for b in actual_blobs if b.endswith('.json')]
# 同日複数ファイルがある場合は最新（タイムスタンプ降順）のみ使用
_by_date: dict[str, str] = {}
for b in sorted(actual_blobs):
    # actual_YYYYMMDD_HHMMSS.json → date = YYYYMMDD
    parts = b.rsplit('/', 1)[-1].replace('.json', '').split('_')
    _date_key = parts[1] if len(parts) >= 2 else b
    _by_date[_date_key] = b  # 後勝ち = 最新タイムスタンプ
actual_blobs = list(_by_date.values())
print(f'  actual files: {len(actual_blobs)} (unique dates)')

all_records: list[dict] = []
for blob_name in actual_blobs:
    data = gcs_load_json(f'gs://{GCS_BUCKET_NAME}/{blob_name}')
    if data and 'actuals' in data:
        for rec in data['actuals']:
            rec['predict_date'] = data.get('predict_date', '')
            rec['actual_date'] = data.get('actual_date', '')
            all_records.append(rec)

df_all = pd.DataFrame(all_records)
print(f'  合計レコード: {len(df_all)}')

if df_all.empty:
    print('No actual data available yet. Run cells 5-6 first.')
else:
    # actual_return を数値に変換（文字列 "+1.23%" の場合もあるため）
    df_all['actual_return'] = pd.to_numeric(df_all['actual_return'], errors='coerce')
    df_valid = df_all.dropna(subset=['actual_return'])

    # ── 4-2. 方向一致率 ──
    dir_acc_total: float = float(df_valid['direction_match'].mean())
    print(f'\n=== 精度集計（全期間） ===')
    print(f'有効レコード: {len(df_valid)}')
    print(f'方向一致率: {dir_acc_total:.1%}')

    # スコア×リターン相関
    corr: float = float(df_valid[['score', 'actual_return']].corr().iloc[0, 1])
    print(f'スコア x リターン相関: {corr:.3f}')

    # ── 4-3. カテゴリ別平均リターン ──
    cat_order = ['DOWN', 'NEUTRAL', 'UP']
    cat_stats = df_valid.groupby('prediction').agg(
        count=('actual_return', 'size'),
        mean_return=('actual_return', 'mean'),
        median_return=('actual_return', 'median'),
        direction_accuracy=('direction_match', 'mean'),
    ).reindex(cat_order).dropna(how='all')

    print('\nカテゴリ別統計:')
    cat_display = cat_stats.copy()
    cat_display['mean_return'] = cat_display['mean_return'].apply(lambda x: f'{x:+.2%}')
    cat_display['median_return'] = cat_display['median_return'].apply(lambda x: f'{x:+.2%}')
    cat_display['direction_accuracy'] = cat_display['direction_accuracy'].apply(lambda x: f'{x:.1%}')
    display(cat_display)

    # ── 4-4. カテゴリ別平均リターン棒グラフ ──
    fig, ax = plt.subplots(figsize=(8, 5))
    colors = ['#d32f2f', '#9e9e9e', '#2e7d32']
    bars = ax.bar(
        cat_stats.index,
        cat_stats['mean_return'] * 100,
        color=[colors[cat_order.index(c)] for c in cat_stats.index],
        edgecolor='black',
        linewidth=0.5,
    )
    ax.axhline(y=0, color='black', linewidth=0.8)
    ax.set_ylabel('平均リターン (%)')
    ax.set_xlabel('予測カテゴリ')
    ax.set_title(f'予測カテゴリ別 翌日平均リターン (n={len(df_valid)})')

    # Add count labels on bars
    for bar, (_, row) in zip(bars, cat_stats.iterrows()):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(),
                f'n={int(row["count"])}', ha='center', va='bottom', fontsize=9)

    plt.tight_layout()
    plt.show()

    # ── 4-5. 日別方向一致率の推移 ──
    if df_valid['predict_date'].nunique() > 1:
        daily = df_valid.groupby('predict_date').agg(
            count=('direction_match', 'size'),
            accuracy=('direction_match', 'mean'),
            mean_return=('actual_return', 'mean'),
        ).reset_index()

        fig, ax1 = plt.subplots(figsize=(10, 5))
        ax1.bar(daily['predict_date'], daily['accuracy'] * 100,
                color='steelblue', alpha=0.7, label='方向一致率')
        ax1.set_ylabel('方向一致率 (%)')
        ax1.set_xlabel('予測日')
        ax1.axhline(y=50, color='red', linestyle='--', linewidth=0.8, label='50%ライン')
        ax1.legend(loc='upper left')
        ax1.set_title('日別 方向一致率の推移')
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        plt.show()

    # ── 4-6. サマリーを GCS に保存 ──
    summary: dict = {
        'updated_at': datetime.now(tz=JST).isoformat(),
        'total_records': len(df_valid),
        'num_dates': int(df_valid['predict_date'].nunique()),
        'direction_accuracy': dir_acc_total,
        'score_return_correlation': corr,
        'category_stats': {
            cat: {
                'count': int(row['count']),
                'mean_return': float(row['mean_return']),
                'median_return': float(row['median_return']),
                'direction_accuracy': float(row['direction_accuracy']),
            }
            for cat, row in cat_stats.iterrows()
        },
    }
    summary_path = f'{GCS_ACCURACY}/accuracy_summary.json'
    gcs_save_json(summary, summary_path)
    print(f'\n精度サマリーを GCS に保存しました: {summary_path}')